In [44]:
from keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from project_brain_decoder.config import get_project_root
from project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import numpy as np
import gc

In [29]:
tf.random.set_seed(42)
np.random.seed(42)

In [30]:
folder = get_project_root() / "data" / "raw"
files = list(folder.glob("*.nwb"))
batch_size, window_size, input_dim = 128, 30, 192

In [31]:
train = files[:187] # 60%
val = files[187:249] # 20%
test = files[249:] # 20%

In [41]:
X_list = []
y_list = []
X_val_list = []
y_val_list = []
X_test_list = []
y_test_list = []

In [33]:
# session = load_nwb(file_path=train[0])["target_mrs_velocity"]
# session.shape

In [34]:
# Train set
for file in train:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    X_list.append(neural)
    y_list.append(targets)


X_train = np.concatenate(X_list, axis=0) # stacking sessions vertically
y_train = np.concatenate(y_list, axis=0)

del X_list, y_list
gc.collect()

38380

In [42]:
# Validation set
for file in val:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    X_val_list.append(neural)
    y_val_list.append(targets)


X_val_train = np.concatenate(X_val_list, axis=0) # stacking sessions vertically
y_val_train = np.concatenate(y_val_list, axis=0)

del X_val_list, y_val_list
gc.collect()

10937

In [43]:
# Test set
for file in test:
    session = load_nwb(file_path=file)
    spiking_band = session["neural_spiking_band"]
    threshold_crossings = session["neural_threshold_crossings"]
    index_velocity = session["target_index_velocity"]
    mrs_velocity = session["target_mrs_velocity"]
    neural = np.concatenate([spiking_band, threshold_crossings], axis=1) # 192 columns
    targets = np.column_stack([index_velocity, mrs_velocity]) # 2 columns
    X_test_list.append(neural)
    y_test_list.append(targets)


X_test_train = np.concatenate(X_test_list, axis=0) # stacking sessions vertically
y_test_train = np.concatenate(y_test_list, axis=0)

del X_test_list, y_test_list
gc.collect()

8782

In [37]:
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()
neural_scaler.fit(X_train)
targets_scaler.fit(y_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [38]:
def get_transformer(window_size, input_dim):
    input_layer = Input(shape=(window_size, input_dim))
    attention_1 = MultiHeadAttention(num_heads=4, key_dim=48)(input_layer, input_layer)
    attention_1 = Dropout(0.1)(attention_1)
    attention_1 = LayerNormalization()(attention_1 + input_layer) # skip connection
    # Feed forward block
    dense_1 = Dense(units=384, activation="relu")(attention_1)
    dense_2 = Dense(units=192)(dense_1)
    attention_2 = Dropout(0.1)(dense_2)
    attention_2 = LayerNormalization()(attention_1 + attention_2) # skip connection 2
    avg_pool = GlobalAveragePooling1D()(attention_2)
    output = Dense(units=2)(avg_pool)
    model = Model(inputs=[input_layer], outputs=[output])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss="mse")
    return model

In [ ]:
def main(model):
    get_transformer(window_size, input_dim).fit(X_train, y_train, batch_size=batch_size, epochs=1, validation_data=(X_val_train, y_val_train), callbacks=[])